In [ ]:
!pip install catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 15.2 MB/s eta 0:00:00


In [ ]:
from lightgbm import early_stopping, log_evaluation
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

In [ ]:
# Google Drive Mount
# from google.colab import drive
# drive.mount("/content/drive")

import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

###############################################################################
# 1) 전역 설정 ------------------------------------------------------------------
###############################################################################
AVG_FILL_FEATURES = [
    "wind_speed", "wind_direction", "visibility", "vapor_pressure",
    "surface_temp", "sea_level_pressure", "humidity", "dew_point"
]
ZERO_FILL_FEATURES = ["sunshine_duration", "snow_depth", "precipitation"]
CLOUD_FEATURE = "cloud_cover"

SCALE_BASE_FEATURES = [
    "dew_point", "humidity", "local_pressure", "precipitation",
    "sea_level_pressure", "snow_depth", "surface_temp", "vapor_pressure",
    "visibility", "wind_direction", "wind_speed", "climatology_temp"
]

# 계절 구분용
IS_WARM = lambda m: 4 <= m <= 9

###############################################################################
# 2) 보간 함수 ------------------------------------------------------------------
###############################################################################
def fill_block_interp(values, *, integer=False):
    n, i = len(values), 0
    while i < n:
        if np.isnan(values[i]):
            s = i
            while i < n and np.isnan(values[i]):
                i += 1
            e = i - 1
            L = e - s + 1

            prev_val = values[s - 1] if s - 1 >= 0 else np.nan
            next_val = values[e + 1] if e + 1 < n else np.nan

            if np.isnan(prev_val) and np.isnan(next_val):
                fill_vals = [np.nan] * L
            elif np.isnan(prev_val):
                fill_vals = [next_val] * L
            elif np.isnan(next_val):
                fill_vals = [prev_val] * L
            else:
                step = (next_val - prev_val) / (L + 1)
                fill_vals = [prev_val + step * (k + 1) for k in range(L)]

            if integer:
                fill_vals = [np.floor(v) if not np.isnan(v) else v for v in fill_vals]
            values[s : e + 1] = fill_vals
        else:
            i += 1
    return values

###############################################################################
# 3) 전처리 + 파생 feature 생성 --------------------------------------------------
###############################################################################
def preprocess(df, is_train=True):
    df = df.replace(-9999, np.nan).copy()

    # 1. 보간 처리
    for feat in [CLOUD_FEATURE] + AVG_FILL_FEATURES:
        cols = [f"{feat}_{h}" for h in range(24)]
        if is_train:
            df = df[df[cols].isna().sum(axis=1) < 12].reset_index(drop=True)
        df[cols] = df[cols].apply(lambda r: fill_block_interp(r.values.copy(), integer=(feat==CLOUD_FEATURE)), axis=1, result_type="expand")
        # 시간 평균/표준편차/최대/최소 파생
        df[f"{feat}_mean"] = df[cols].mean(axis=1)
        df[f"{feat}_std"]  = df[cols].std(axis=1)
        df[f"{feat}_max"]  = df[cols].max(axis=1)
        df[f"{feat}_min"]  = df[cols].min(axis=1)

    # 2. 결측값 0 채움
    for feat in ZERO_FILL_FEATURES:
        cols = [f"{feat}_{h}" for h in range(24)]
        df[cols] = df[cols].fillna(0)

    # 3. 시간 기반 추가 파생 feature
    for feat in ["surface_temp", "humidity", "dew_point", "wind_speed"]:
      cols = [f"{feat}_{h}" for h in range(24) if f"{feat}_{h}" in df.columns]

      if len(cols) == 24:
          # 하루 내 변화량
          df[f"{feat}_diff"] = df[cols[-1]] - df[cols[0]]

          # 오전 vs 오후 평균
          df[f"{feat}_am_mean"] = df[cols[:12]].mean(axis=1)
          df[f"{feat}_pm_mean"] = df[cols[12:]].mean(axis=1)

          # 전체 증가 방향성 (시계열 변화량의 합)
          df[f"{feat}_trend"] = df[cols].diff(axis=1).mean(axis=1)

    # 4. 날짜 기반 파생
    df["month"] = pd.to_datetime("2023-" + df["date"], format="%Y-%m-%d", errors='coerce').dt.month
    df["is_warm"] = df["month"].apply(IS_WARM)

    drop_cols = ["date", "station", "station_name", "min_cloud_height", "wind_direction"]
    if not is_train and "target" in df.columns:
        drop_cols.append("target")
    return df.drop(columns=[c for c in drop_cols if c in df.columns])

###############################################################################
# 4) 데이터 로딩 및 전처리 --------------------------------------------------------
###############################################################################
# PATH = "/content/drive/MyDrive/25-1 머신런링/#TP_1/next-day-air-temperature-forecast-challenge/"
train_raw = pd.read_csv("/content/train_dataset.csv")
test_raw  = pd.read_csv("/content/test_dataset.csv")

train = preprocess(train_raw, is_train=True)
test  = preprocess(test_raw, is_train=False)

###############################################################################
# 5) Scaling ----------------------------------------------------------------------
###############################################################################
scale_cols = [f"{b}_{h}" for b in SCALE_BASE_FEATURES for h in range(24) if f"{b}_{h}" in train.columns]
mean_std = {c: (train[c].mean(skipna=True), train[c].std(skipna=True)) for c in scale_cols}
for c, (mu, sig) in mean_std.items():
    sig = sig if (sig != 0 and not np.isnan(sig)) else 1.0
    train[c] = (train[c] - mu) / sig
    test[c]  = (test[c] - mu) / sig

###############################################################################
# 6) wet/dry 분리 ------------------------------------------------------------------
###############################################################################
prec_cols = [f"precipitation_{h}" for h in range(24)]
snow_cols = [f"snow_depth_{h}" for h in range(24)]

wet_tr_idx = (train[prec_cols + snow_cols] > 0).any(axis=1)
wet_te_idx = (test[prec_cols + snow_cols]  > 0).any(axis=1)
dry_tr_idx = ~wet_tr_idx
dry_te_idx = ~wet_te_idx

wet_tr = train[wet_tr_idx]
dry_tr = train[dry_tr_idx]

X_wet = wet_tr.drop(columns=["target", "id"])
y_wet = wet_tr["target"]
X_dry = dry_tr.drop(columns=["target", "id"])
y_dry = dry_tr["target"]
X_test_wet = test.loc[wet_te_idx].drop(columns=["id"])
X_test_dry = test.loc[dry_te_idx].drop(columns=["id"])

###############################################################################
# 7) 모델 + 앙상블 -----------------------------------------------------------------
###############################################################################
def make_lgbm():
    return LGBMRegressor(n_estimators= 5000, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=42)

def make_stack():
    return StackingRegressor(
        estimators=[
            ("lgb", make_lgbm()),
            ("xgb", XGBRegressor(n_estimators=500, learning_rate=0.03, random_state=42, verbosity=0)),
            ("cat", CatBoostRegressor(n_estimators=500, learning_rate=0.03, verbose=0, random_state=42)),
        ],
        final_estimator=Ridge(alpha=1.0),
        n_jobs=-1
    )

scores = cross_val_score(make_stack(), X_wet, y_wet, cv=5, scoring='neg_root_mean_squared_error')
print(f"RMSE 평균: {-np.mean(scores):.4f}")

scores = cross_val_score(make_stack(), X_dry, y_dry, cv=5, scoring='neg_root_mean_squared_error')
print(f"RMSE 평균: {-np.mean(scores):.4f}")

model_wet = make_stack().fit(X_wet, y_wet)
model_dry = make_stack().fit(X_dry, y_dry)

###############################################################################
# 8) 예측 및 저장 ------------------------------------------------------------------
###############################################################################
preds = pd.Series(index=test.index, dtype=float)
preds[wet_te_idx] = model_wet.predict(X_test_wet)
preds[dry_te_idx] = model_dry.predict(X_test_dry)

submission = pd.DataFrame({"id": test["id"], "target_pred": preds})
submission.to_csv("submission.csv", index=False)
print("최종 예측 완료! → 'submission.csv' 저장")
print(submission.head())


<ipython-input-3-f5469b34562a>:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{feat}_diff"] = df[cols[-1]] - df[cols[0]]
<ipython-input-3-f5469b34562a>:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{feat}_am_mean"] = df[cols[:12]].mean(axis=1)
<ipython-input-3-f5469b34562a>:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmente

RMSE 평균: 1.3937


/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


최종 예측 완료! → 'submission.csv' 저장
   id  target_pred
0   0     1.887646
1   1     0.855555
2   2     1.234925
3   3     0.660848
4   4     1.126628
